# MicroPlant — Model Training

This notebook trains the MicroPlant model for plant disease classification.

We explore:
- Baseline training (MicroPlant)
- Teacher model (ResNet18)
- Knowledge Distillation (KD)

The goal is to build an accurate yet lightweight model suitable for edge deployment.

In [9]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.preprocessing import get_dataloaders
from src.architectures import get_microplant, get_teacher_model
from src.training import train_model, validate
from src.utils import count_parameters, count_model_bytes

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="torch.utils.data.dataloader")

## Configuration

We define training parameters and device setup.

In [15]:
DATA_DIR = "../data/MicroPlantDataset"

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

BATCH_SIZE = 64
EPOCHS = 10
LR = 0.001

print(f"using : {DEVICE}")

using : cpu


## Load Dataset

We use the predefined data pipeline with augmentation and proper splitting.

In [11]:
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    DATA_DIR, batch_size=BATCH_SIZE
)

print("Classes:", class_names)

Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy']


## Baseline Model (MicroPlant)

We first train the MicroPlant model without any distillation.

This serves as a baseline for comparison.

In [12]:
baseline_model = get_microplant(num_classes=len(class_names)).to(DEVICE)

print("Parameters:", count_parameters(baseline_model))
count_model_bytes(baseline_model)

Parameters: 8804
Model Weights: 35,216 bytes
Model Buffers: 2,648 bytes
Total Size: 36.98 KB


37864

In [21]:
baseline_model = train_model(
    baseline_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    save_name="../models/microplant_baseline",
    device=DEVICE
)

Training: 100%|██████████| 41/41 [00:26<00:00,  1.57it/s]


Epoch 1 | Train Loss 0.4493 Acc 0.86 | Val Loss 0.3165 F1 0.9018


Training: 100%|██████████| 41/41 [00:23<00:00,  1.76it/s]


Epoch 2 | Train Loss 0.3751 Acc 0.89 | Val Loss 0.2497 F1 0.9160


Training: 100%|██████████| 41/41 [00:24<00:00,  1.69it/s]


...

Epoch 8 | Train Loss 0.2016 Acc 0.94 | Val Loss 0.1237 F1 0.9616


Training: 100%|██████████| 41/41 [00:23<00:00,  1.73it/s]


Epoch 9 | Train Loss 0.1695 Acc 0.94 | Val Loss 0.1134 F1 0.9581


Training: 100%|██████████| 41/41 [00:21<00:00,  1.93it/s]


Epoch 10 | Train Loss 0.1546 Acc 0.95 | Val Loss 0.1965 F1 0.9313


In [22]:
baseline_loss, baseline_f1 = validate(baseline_model, test_loader, device=DEVICE)

print("Baseline F1:", baseline_f1)

Baseline F1: 0.9469189866175336


## Teacher Model (ResNet18)

We train a larger model to act as a teacher.

This model provides strong supervision for distillation.

In [23]:
teacher_model = get_teacher_model(num_classes=len(class_names)).to(DEVICE)

print("Teacher parameters:", count_parameters(teacher_model))

KeyError: 'pretrained=True'

In [ ]:
teacher_model = train_model(
    teacher_model,
    train_loader,
    val_loader,
    epochs=10,
    lr=LR,
    save_name="../models/teacher",
    device=DEVICE
)

In [ ]:
teacher_loss, teacher_f1 = validate(teacher_model, test_loader, device=DEVICE)

print("Teacher F1:", teacher_f1)

## Knowledge Distillation

We train the MicroPlant model using guidance from the teacher model.

The student learns from:
- Ground truth labels
- Teacher predictions (soft targets)

In [19]:
student_model = get_microplant(num_classes=len(class_names)).to(DEVICE)

In [ ]:
student_model = train_model(
    student_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    teacher=teacher_model,
    kd_alpha=0.7,
    kd_temp=3.0,
    lr=LR,
    save_name="../models/microplant_kd",
    device=DEVICE
)

In [ ]:
student_loss, student_f1 = validate(student_model, test_loader, device=DEVICE)

print("Student (KD) F1:", student_f1)

## Model Comparison

We compare the performance of:
- Baseline MicroPlant
- Teacher (ResNet18)
- Distilled MicroPlant

In [20]:
print("=== Final Results ===")
print(f"Baseline F1: {baseline_f1:.4f}")
print(f"Teacher F1:  {teacher_f1:.4f}")
print(f"Student F1:  {student_f1:.4f}")

=== Final Results ===
Baseline F1: 0.8793


NameError: name 'teacher_f1' is not defined

## Observations

- The teacher model achieves the highest performance due to its capacity
- The distilled student improves over the baseline model
- Knowledge distillation helps transfer knowledge into a smaller model

### Conclusion
The MicroPlant model benefits from distillation, achieving strong performance while remaining lightweight and suitable for edge deployment.